# f696原模型低beta但接受目标顶住cap日的账户分量

只读模型权重、接受目标、原生0bp Tiered账户日级现金/SOXL/费用分量；按预定216/243日对照和819窗口报告。该路径归因不是调整平滑的账户反事实。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_f696_raw_below_cap_path_loss_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
a=root/s['action_card'];f=root/s['A0_card'];b=root/s['account_card']
for p,key in [(root/s['trace_plan'],'trace_plan_sha256'),(root/'research/stage1_f696_beta_saturation_layer_trace_20261024/summary.json','trace_summary_sha256'),(a/'summary.json','action_summary_sha256'),(f/'summary.json','A0_summary_sha256'),(b/'summary.json','account_summary_sha256')]:assert sha(p)==s[key],key
assert sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
sa=json.loads((a/'summary.json').read_text());sf=json.loads((f/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sa['nas_evidence']['sha256']==s['action_NAS_sha256'] and sf['nas_evidence']['sha256']==s['A0_NAS_sha256']
assert sb['nas_readback']['verified'] and sb['nas_readback']['sha256']==s['account_NAS_sha256']
for source,exp,rid,art,key in [(a,'action_experiment_id','action_recorder_id','inputs/raw_predictions','raw_predictions_sha256'),(a,'action_experiment_id','action_recorder_id','inputs/source_targets','source_targets_sha256'),(f,'A0_experiment_id','A0_recorder_id','inputs/frame','A0_frame_sha256'),(f,'A0_experiment_id','A0_recorder_id','native/A_active_research_replay_0','A0_native_sha256'),(b,'account_experiment_id','account_recorder_id','analysis/daily','account_daily_sha256'),(b,'account_experiment_id','account_recorder_id','analysis/rolling126','account_rolling_sha256')]:
 assert sha(source/'qlib_recorder'/s[exp]/s[rid]/'artifacts'/art)==s[key],key
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['action_experiment_id'],recorder_id=s['action_recorder_id']);raw_dict=rec.load_object('inputs/raw_predictions');accepted_dict=rec.load_object('inputs/source_targets')
with R.uri_context(str((f/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['A0_experiment_id'],recorder_id=s['A0_recorder_id']);frame=rec.load_object('inputs/frame').reset_index(drop=True);native=rec.load_object('native/A_active_research_replay_0')
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['account_experiment_id'],recorder_id=s['account_recorder_id']);daily=rec.load_object('analysis/daily').reset_index(drop=True);rolling=rec.load_object('analysis/rolling126')
signal=pd.DatetimeIndex(frame.signal_date);trade=pd.DatetimeIndex(frame.trade_date)
assert len(frame)==len(daily)==944 and len(rolling)==819
assert trade.equals(pd.DatetimeIndex(daily.trade_date)) and signal.equals(pd.DatetimeIndex(daily.signal_date))
assert signal.isin(pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'))).all()
raw=raw_dict['frozen_current'].reindex(signal);accepted=accepted_dict['frozen_current'].reindex(signal)
columns=['soxl_weight','soxx_weight','cash_weight','tail_probability','alpha_probability']
np.testing.assert_allclose(accepted[columns].to_numpy(float),frame[columns].to_numpy(float),atol=1e-7,rtol=0)
auth=native['authority'].reset_index(drop=True);targets=native['targets'].reset_index(drop=True)
assert pd.DatetimeIndex(auth.trade_date).equals(trade) and pd.DatetimeIndex(targets.trade_date).equals(trade)
execute=auth.execute_open_target.to_numpy(bool);cap=auth.current_beta_cap.to_numpy(float)
raw_beta=3*raw.soxl_weight.to_numpy(float)+raw.soxx_weight.to_numpy(float)
accepted_beta=3*accepted.soxl_weight.to_numpy(float)+accepted.soxx_weight.to_numpy(float)
projected_beta=3*targets.soxl_weight.to_numpy(float)+targets.soxx_weight.to_numpy(float)
primary=execute&(accepted_beta<=cap+1e-9)&(cap-projected_beta<=1e-10)
raw_below=primary&(raw_beta<cap-1e-10);raw_high=primary&(raw_beta>=cap-1e-10)
assert (int(primary.sum()),int(raw_below.sum()),int(raw_high.sum()))==(459,216,243)
group=np.where(raw_below,'raw_below_accepted_cap',np.where(raw_high,'raw_high_accepted_cap',np.where(execute,'other_execute','HOLD')))
components=['paired_log_excess','cash_opportunity_log','SOXL_selection_log','Stage1_direct_fees_log','timing_and_benchmark_residual_log']
assert np.isfinite(daily[components].to_numpy(float)).all()
closure=(daily[components[1:]].sum(axis=1)-daily.paired_log_excess).abs();assert float(closure.max())<1e-10
x=daily[['trade_date']+components].copy();x['trade_date']=trade;x['year']=trade.year;x['quarter']=trade.quarter;x['group']=group
names=['raw_below_accepted_cap','raw_high_accepted_cap','other_execute','HOLD']
def stats(g):
 out={'days':len(g)}
 for c in components:
  out[c+'_sum']=float(g[c].sum());out[c+'_mean']=float(g[c].mean()) if len(g) else None
 for c in ('paired_log_excess','cash_opportunity_log','SOXL_selection_log'):
  out[c+'_positive_days']=int((g[c]>0).sum());out[c+'_negative_days']=int((g[c]<0).sum())
 return out
all_group={name:stats(x[x.group==name]) for name in names}
annual=[{'year':int(y),'partial_endpoint':y in (2023,2026),'groups':{name:stats(g[g.group==name]) for name in names}} for y,g in x.groupby('year',sort=True)]
quarter=[{'year':int(y),'quarter':int(q),'groups':{name:stats(g[g.group==name]) for name in names}} for (y,q),g in x.groupby(['year','quarter'],sort=True)]
assert len(annual)==4 and len(quarter)==16 and sum(v['days'] for v in all_group.values())==944
windows=[]
for i in range(819):
 g=x.iloc[i:i+126];rw=rolling.iloc[i];assert pd.Timestamp(g.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 outcome='weak' if rw.relative_log_excess<0 else 'good' if rw.relative_log_excess>0 else 'tie'
 a0=g[g.group==names[0]];b0=g[g.group==names[1]]
 windows.append([str(g.trade_date.iloc[0].date()),outcome,len(a0),len(b0),float(a0.cash_opportunity_log.sum()),float(a0.SOXL_selection_log.sum()),float(a0.paired_log_excess.sum()),float(b0.cash_opportunity_log.sum()),float(b0.SOXL_selection_log.sum()),float(b0.paired_log_excess.sum())])
w=pd.DataFrame(windows,columns=['start','outcome','raw_below_days','raw_high_days','raw_below_cash_sum','raw_below_SOXL_sum','raw_below_excess_sum','raw_high_cash_sum','raw_high_SOXL_sum','raw_high_excess_sum'])
assert len(w)==819 and w.outcome.value_counts().to_dict()=={'weak':564,'good':255}
window_summary={state:{'windows':len(g),'mean_raw_below_days':float(g.raw_below_days.mean()),'mean_raw_high_days':float(g.raw_high_days.mean()),
                       'mean_raw_below_cash_sum':float(g.raw_below_cash_sum.mean()),'mean_raw_below_SOXL_sum':float(g.raw_below_SOXL_sum.mean()),
                       'mean_raw_below_excess_sum':float(g.raw_below_excess_sum.mean())} for state,g in w.groupby('outcome')}
result={'status':'read_only_f696_raw_below_accepted_cap_frozen_path_components_no_policy','plan_sha256':sha(plan_path),
        'source_recorder_ids':{'action':s['action_recorder_id'],'A0':s['A0_recorder_id'],'account':s['account_recorder_id']},
        'full_sessions':len(x),'primary_days':int(primary.sum()),'raw_below_days':int(raw_below.sum()),'raw_high_days':int(raw_high.sum()),
        'daily_component_max_closure_error':float(closure.max()),'all_groups':all_group,'annual':annual,'quarter':quarter,
        'rolling126_columns':list(w.columns),'rolling126':windows,'rolling126_summary':window_summary,
        'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['Frozen-path cash/SOXL attribution is not the counterfactual impact of shortening EMA/HOLD','Group membership uses predecision source weights and same-day own-state cap but weak/good windows use future performance','Overlapping windows and reused history cannot select a tradable switch']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'groups':{k:{x:v[x] for x in ['days','paired_log_excess_sum','cash_opportunity_log_sum','SOXL_selection_log_sum']} for k,v in all_group.items()},
       'annual_raw_below':[(z['year'],z['groups'][names[0]]['days'],z['groups'][names[0]]['paired_log_excess_sum'],z['groups'][names[0]]['cash_opportunity_log_sum'],z['groups'][names[0]]['SOXL_selection_log_sum']) for z in annual],
       'windows':window_summary})


{'groups': {'raw_below_accepted_cap': {'days': 216, 'paired_log_excess_sum': -0.01623131102991044, 'cash_opportunity_log_sum': -0.11801449899031513, 'SOXL_selection_log_sum': 0.10191118542780478}, 'raw_high_accepted_cap': {'days': 243, 'paired_log_excess_sum': -0.10073520262270075, 'cash_opportunity_log_sum': -0.046652940659438785, 'SOXL_selection_log_sum': -0.051124429106842374}, 'other_execute': {'days': 338, 'paired_log_excess_sum': -0.14837581145582868, 'cash_opportunity_log_sum': -0.28752789213316754, 'SOXL_selection_log_sum': 0.13990357337092305}, 'HOLD': {'days': 147, 'paired_log_excess_sum': -0.11152594291728024, 'cash_opportunity_log_sum': -0.1481272691206953, 'SOXL_selection_log_sum': 0.035241128534510585}}, 'annual_raw_below': [(2023, 58, -0.01983392137263376, 0.036348870743647156, -0.05516522260401108), (2024, 88, -0.04665403095413749, -0.09576840076522308, 0.04930399554702613), (2025, 44, 0.02028699197880861, -0.06624428825701609, 0.08565185923988612), (2026, 26, 0.0299696

## 中文结论

# 原模型低beta但接受目标顶cap：损失分量并非统一方向

只读精确f696原模型输出、EMA/HOLD接受目标、原生0bp Tiered资金账户日级现金/SOXL/费用分量。原始、接受、风险投影三层按944个Qlib信号/交易日严格对齐；459个“接受目标不超cap且投影无余量”的原开盘执行日中，原模型低于cap **216日**、原模型达到或超过cap **243日**。日级成分对SOXX对数超额闭合误差最大 **1.04e−17**；四年（2023/2026部分）、16季、全部819个连续126日窗口均保留。0拟合、新账户或阈值搜索。

原模型低beta的216日，原账户对SOXX累计对数超额 **−0.01623**，其中现金机会 **−0.11801**、SOXL相对选择 **+0.10191**，两项大体抵消。2023部分年58日现金贡献 **+0.03635**（有保护）、SOXL选择 **−0.05517**；2024完整年88日现金 **−0.09577**、SOXL选择 **+0.04930**；2025完整年44日现金 **−0.06624**、SOXL选择 **+0.08565**；2026部分年26日现金 **+0.00765**、SOXL选择 **+0.02212**。原模型高beta的243日全段超额 **−0.10074**，但2023主要现金拖累、2025主要SOXL选择拖累，也不是单一损失模式。

原账户胜SOXX的255个126日窗口平均含 **36.6** 个原模型低beta/接受顶cap日，未胜的564窗仅 **26.55** 个；这些窗口高度重叠且按未来收益标记。不能据此把这216日统一改成原始模型低beta目标，或把缩短EMA/HOLD称为因子改进。当前同路径贡献不是改变目标后的自身状态、费用和风险反馈反事实。

决定 `descriptive_raw_below_cap_smoothing_group_mixed_components_no_policy`。配置链上的滞后确实会让原模型低beta日继续顶cap，但它在不同年份有相反的现金保护与SOXL选择含义。下一可证伪机制需在决策前同时区分现金保护是否有价值及SOXL相对选择是否有利，然后冻结接收层、完整0bp Tiered自身状态账户和全部窗口门；不能从这张事后分组卡挑年份或调平滑参数。SOXX v2目标未达成。
